In [1]:
import pandas as pd
import numpy as np
import random
from faker import Faker

fake = Faker("en_IN")

random.seed(42)
np.random.seed(42)
Faker.seed(42)



In [2]:
customers_df = pd.read_csv("Customers1.csv")
customers_df.head()

ACCOUNT_TYPES = ["Savings", "Current"]
ACCOUNT_WEIGHTS = [80, 20]

STATUS = ["Active", "Inactive", "Closed"]
STATUS_WEIGHTS = [90, 7, 3]

BRANCH_CITIES = [
    "Kolkata",
    "Delhi",
    "Mumbai",
    "Bengaluru",
    "Hyderabad",
    "Chennai",
    "Pune",
    "Ahmedabad",
    "Jaipur",
    "Lucknow"
]

BRANCH_CODES = [f"BR{i}" for i in range(1001,1031)]

customers_with_two_accounts = set(
    random.sample(
        list(customers_df["Customer_ID"]),
        5000
    )
)
accounts = []
account_number = 1

In [5]:
accounts = []
account_number = 1

In [6]:
for _, customer in customers_df.iterrows():

    customer_id = customer["Customer_ID"]

    # Customer Join Date
    join_date = pd.to_datetime(customer["Join_Date"])

    # ---------- First Account ----------

    account_type = random.choices(
        ACCOUNT_TYPES,
        weights=ACCOUNT_WEIGHTS,
        k=1
    )[0]

    branch_city = random.choice(BRANCH_CITIES)

    branch_code = random.choice(BRANCH_CODES)

    if account_type == "Savings":
        balance = random.randint(5000, 1500000)
    else:
        balance = random.randint(10000, 5000000)

    status = random.choices(
        STATUS,
        weights=STATUS_WEIGHTS,
        k=1
    )[0]

    opening_date = fake.date_between(
        start_date=join_date.date(),
        end_date="today"
    )

    account_id = f"A{account_number:09d}"

    account_number += 1

    accounts.append({
        "Account_ID": account_id,
        "Customer_ID": customer_id,
        "Account_Type": account_type,
        "Branch_Code": branch_code,
        "Branch_City": branch_city,
        "Opening_Date": opening_date,
        "Current_Balance": balance,
        "Status": status
    })
        # ---------- Second Account (Only for Selected Customers) ----------

    if customer_id in customers_with_two_accounts:

        # Second account must be different
        if account_type == "Savings":
            second_account_type = "Current"
        else:
            second_account_type = "Savings"

        second_branch_city = random.choice(BRANCH_CITIES)

        second_branch_code = random.choice(BRANCH_CODES)

        if second_account_type == "Savings":
            second_balance = random.randint(5000, 1500000)
        else:
            second_balance = random.randint(10000, 5000000)

        second_status = random.choices(
            STATUS,
            weights=STATUS_WEIGHTS,
            k=1
        )[0]

        second_opening_date = fake.date_between(
            start_date=join_date.date(),
            end_date="today"
        )

        second_account_id = f"A{account_number:09d}"

        account_number += 1

        accounts.append({
            "Account_ID": second_account_id,
            "Customer_ID": customer_id,
            "Account_Type": second_account_type,
            "Branch_Code": second_branch_code,
            "Branch_City": second_branch_city,
            "Opening_Date": second_opening_date,
            "Current_Balance": second_balance,
            "Status": second_status
        })

    

In [7]:
accounts_df = pd.DataFrame(accounts)
accounts_df.shape


(15000, 8)

In [8]:
accounts_df.head()


,Account_ID,Customer_ID,Account_Type,Branch_Code,Branch_City,Opening_Date,Current_Balance,Status
0,A000000001,C00001,Savings,BR1007,Hyderabad,2026-02-18,508838,Active
1,A000000002,C00002,Current,BR1009,Ahmedabad,2022-02-15,1654416,Active
2,A000000003,C00002,Savings,BR1004,Delhi,2023-07-24,726488,Active
3,A000000004,C00003,Savings,BR1005,Lucknow,2026-04-11,1216997,Active
4,A000000005,C00004,Savings,BR1024,Lucknow,2019-09-10,230995,Active


In [9]:
accounts_df["Account_ID"].duplicated().sum()


np.int64(0)

In [10]:
accounts_df.groupby("Customer_ID").size().value_counts()


1    5000
2    5000
Name: count, dtype: int64

In [11]:
accounts_df["Account_Type"].value_counts()


Account_Type
Savings    8985
Current    6015
Name: count, dtype: int64

In [12]:
accounts_df["Status"].value_counts(normalize=True) * 100


Status
Active      89.973333
Inactive     6.993333
Closed       3.033333
Name: proportion, dtype: float64

In [15]:
accounts_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15000 entries, 0 to 14999
Data columns (total 8 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   Account_ID       15000 non-null  object
 1   Customer_ID      15000 non-null  object
 2   Account_Type     15000 non-null  object
 3   Branch_Code      15000 non-null  object
 4   Branch_City      15000 non-null  object
 5   Opening_Date     15000 non-null  object
 6   Current_Balance  15000 non-null  int64 
 7   Status           15000 non-null  object
dtypes: int64(1), object(7)
memory usage: 937.6+ KB


In [16]:
accounts_df["Opening_Date"] = pd.to_datetime(accounts_df["Opening_Date"])

In [17]:
accounts_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15000 entries, 0 to 14999
Data columns (total 8 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   Account_ID       15000 non-null  object        
 1   Customer_ID      15000 non-null  object        
 2   Account_Type     15000 non-null  object        
 3   Branch_Code      15000 non-null  object        
 4   Branch_City      15000 non-null  object        
 5   Opening_Date     15000 non-null  datetime64[ns]
 6   Current_Balance  15000 non-null  int64         
 7   Status           15000 non-null  object        
dtypes: datetime64[ns](1), int64(1), object(6)
memory usage: 937.6+ KB


In [18]:
accounts_df.to_csv(
    "Accounts.csv",
    index=False
)